# File contains complete code of using **ViT**, for classificiation of violent activity, along with **custom YOLOv8** weapon detection model and **scene understanding** model for report generation with the help of **LangChain (OpenAI GPT + Google FLAN-T5)**:

In [1]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.2/413.2 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.33
    Uninstalling langchain-core-0.3.33:
      Successfully uninstalled langchain-core-0.3.33
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.5
    Uninstalling langchain-text-splitters-0.3.5:
      Successfully uninstalled langchain-text-splitters-0.3.5
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.17
    Uninstalling langchain-0.3.17:
      Successfully uninstalled langchain-0.3.17


In [2]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=1430b31bedcdeaa6571a0fbdf815e6e84d1b2e0cd5b80cdf43606b118e853f81
  Stored in directory: /root/.cache/pip/wheels/65/4f/66/bbda9866da446a72e206d6484cd97381cbc7859a7068541c36
Successfully built fpdf


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [4]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 18.4 MB/s eta 0:00:00


In [5]:
import os
import cv2
import math
import torch
import openai
import numpy as np
from PIL import Image
from fpdf import FPDF
import mediapipe as mp
from ultralytics import YOLO
from datetime import datetime
from langchain.llms import OpenAI
from langchain.chains import LLMChain
import torchvision.transforms as transforms
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, ViTForImageClassification, CLIPProcessor, CLIPModel

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# ==================== Setup and Model Initialization ====================

# Initialize OpenAI / LangChain LLM (for report generation)
llm = OpenAI(api_key="API_KEY")  # Replace with your OpenAI API key

# Initialize FLAN-T5 model and tokenizer (fallback)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
flan_model_name = "google/flan-t5-base"
flan_tokenizer = AutoTokenizer.from_pretrained(flan_model_name)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(flan_model_name).to(device)

# Define Image Transformations for ViT
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

<ipython-input-6-e0ad8a312ec0>:4: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  llm = OpenAI(api_key="API_KEY")  # Replace with your OpenAI API key
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
# Class Labels for ViT Model
class_labels = {
    0: 'fighting',
    1: 'abuse',
    2: 'arson',
    3: 'burglary',
    4: 'shooting',
    5: 'vandalism'
}

# Load ViT Model for Activity Recognition
vit_model_path = "/content/vit_anomaly_detector (1).pth"  # Update with your ViT .pth file path
vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224", num_labels=6, ignore_mismatched_sizes=True
)

vit_model.load_state_dict(torch.load(vit_model_path, map_location=device))
vit_model.to(device).eval()

# Load Weapon (and Person) Detection Model (YOLOv8)
weapon_model_path = "/content/best_weapons.pt"  # Update with your weapon detection .pt file
weapon_model = YOLO(weapon_model_path)
weapon_model.to(device).eval()

# Initialize MediaPipe Pose for pose estimation
mp_pose = mp.solutions.pose
pose_detector = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

# Create output folder for saved weapon holder images
output_img_folder = "weapon_holder_images"
os.makedirs(output_img_folder, exist_ok=True)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([6, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# ==================== Utility Functions ====================

def predict_activity(frame):
    """Predicts the activity label for a given frame using the ViT model."""
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = vit_model(image)
        logits = output.logits
        predicted_class = torch.argmax(logits, dim=1).item()
    return class_labels.get(predicted_class, "normal")

def detect_objects(frame):
    """
    Runs the YOLO model on the given frame and returns two lists:
      - weapons: list of bounding boxes for weapons
      - persons: list of bounding boxes for persons
    Bounding boxes are tuples of (x1, y1, x2, y2).
    """
    results = weapon_model(frame)
    weapons, persons = [], []
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf[0].item()
            cls = int(box.cls[0].item())
            if cls == 1 and conf >= 0.75:  # Assuming Class 1 is 'Weapon'
                weapons.append((x1, y1, x2, y2))
            elif cls == 0 and conf >= 0.8:  # Assuming Class 0 is 'Person'
                persons.append((x1, y1, x2, y2))
    return weapons, persons

def identify_weapon_holder(weapons, persons):
    """Returns the bounding box of a person holding a weapon if found."""
    for wx1, wy1, wx2, wy2 in weapons:
        for px1, py1, px2, py2 in persons:
            # If the weapon's top-left is within the person's bounding box
            if px1 < wx1 < px2 and py1 < wy1 < py2:
                return (px1, py1, px2, py2)
    return None

def update_unique_objects(unique_list, detections, threshold=50):
    """
    Updates the list of unique objects based on detection bounding boxes.
    For each detection, the centroid is calculated and compared with centroids in unique_list.
    If the distance is greater than 'threshold' from all existing ones, it is added.
    """
    for box in detections:
        x1, y1, x2, y2 = box
        center = ((x1 + x2) / 2, (y1 + y2) / 2)
        found = False
        for u_center in unique_list:
            distance = math.sqrt((center[0] - u_center[0]) ** 2 + (center[1] - u_center[1]) ** 2)
            if distance < threshold:
                found = True
                break
        if not found:
            unique_list.append(center)

def run_pose_estimation_and_save(crop_img, frame_index):
    """
    Runs MediaPipe Pose estimation on the given image crop,
    draws pose landmarks on it, and saves the image.
    Returns the annotated image.
    """
    crop_rgb = cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)
    results = pose_detector.process(crop_rgb)
    annotated_image = crop_img.copy()
    if results.pose_landmarks:
        mp.solutions.drawing_utils.draw_landmarks(
            annotated_image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    output_path = os.path.join(output_img_folder, f"weapon_holder_frame_{frame_index}.jpg")
    cv2.imwrite(output_path, annotated_image)
    return annotated_image

In [12]:
# ==================== Report Generation Functions ====================

def generate_scene_description_with_openai(activity, num_people, num_weapons):
    """Generates a scene description using OpenAI's API via LangChain."""
    prompt_template = PromptTemplate(
        input_variables=["activity", "num_people", "num_weapons"],
        template=(
            "You are a crime scene investigator analyzing a video. "
            "The detected activity is '{activity}'. "
            "There are {num_people} people involved and {num_weapons} weapons detected. "
            "Write a detailed and professional crime scene report describing the event in a formal manner."
        ),
    )
    prompt = prompt_template.format(activity=activity, num_people=num_people, num_weapons=num_weapons)
    response = llm.predict(prompt)
    return response

def generate_scene_description_with_flan(activity, num_people, num_weapons):
    """Generates a scene description using FLAN-T5."""
    prompt = (
        f"You are a crime scene investigator analyzing a video. "
        f"The detected activity is '{activity}'. "
        f"There are {num_people} people involved and {num_weapons} weapons detected. "
        f"Write a detailed and professional crime scene report of at least 4 to 6 lines, describing the event in a formal manner so that it may be helpful in a police investigation."
    )
    inputs = flan_tokenizer(prompt, return_tensors="pt").to(device)
    outputs = flan_model.generate(**inputs, max_length=1000)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_scene_description(activity, num_people, num_weapons):
    """Generates a scene description using OpenAI's API, with FLAN-T5 as a fallback."""
    try:
        return generate_scene_description_with_openai(activity, num_people, num_weapons)
    except Exception as e:
        print(f"OpenAI API error: {e}. Falling back to FLAN-T5.")
        return generate_scene_description_with_flan(activity, num_people, num_weapons)

def generate_pdf_report(activity, scene_description, num_people, num_weapons, output_path):
    """
    Generates a formal crime scene report PDF in a custom format.
    The layout includes:
      - Title ("Crime Scene Report")
      - Date, Time, and Location fields
      - Detected Crime, People Involved, Weapons Found
      - A multi-line scene description
    """
    pdf = FPDF('P', 'mm', 'A4')
    pdf.add_page()

    # Report Title
    pdf.set_font("Arial", 'B', 20)
    pdf.cell(0, 10, "Crime Scene Report", ln=True, align="C")
    pdf.ln(5)

    # Horizontal line
    pdf.set_line_width(0.5)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(10)

    # Date, Time, and Location
    now = datetime.now()
    date_str = now.strftime("%d-%m-%Y")
    time_str = now.strftime("%H:%M:%S")
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(40, 10, "Date:")
    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, date_str, ln=True)

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(40, 10, "Time:")
    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, time_str, ln=True)

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(40, 10, "Location:")
    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, "Not Specified", ln=True)  # Placeholder

    pdf.ln(5)

    # Crime Details
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(60, 10, "Detected Crime:")
    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, activity, ln=True)

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(60, 10, "People Involved:")
    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, str(num_people), ln=True)

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(60, 10, "Weapons Found:")
    pdf.set_font("Arial", '', 12)
    pdf.cell(0, 10, str(num_weapons), ln=True)

    pdf.ln(10)

    # Scene Description Header
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, "Scene Description:", ln=True)
    pdf.ln(5)
    pdf.set_font("Arial", '', 12)
    pdf.multi_cell(0, 8, scene_description)

    pdf.ln(10)
    # Footer with generation timestamp
    pdf.set_font("Arial", 'I', 10)
    pdf.cell(0, 10, f"Report generated on {date_str} at {time_str}", align="C")

    pdf.output(output_path)

In [13]:
# ==================== Video Processing ====================

video_path = "/content/Vandalism020_x264.mp4"  # Update with your video path
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
output_video_path = "/content/output_video.mp4"
fourcc = cv2.VideoWriter_fourcc(*'avc1')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# For unique detection we use lists to store the centroid of each unique detection.
unique_persons = []   # List of centers (x, y) for unique persons
unique_weapons = []   # List of centers (x, y) for unique weapons

# For activity counts (if needed)
activity_counts = {label: 0 for label in class_labels.values()}

# Variable for persistent weapon holder overlay:
weapon_holder_crop = None  # Stores the cropped & pose–annotated image of the person holding a weapon

frame_index = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_index += 1

    # Detect activity
    detected_activity = predict_activity(frame)
    activity_counts[detected_activity] += 1

    # Detect objects (weapons and persons)
    weapons, persons = detect_objects(frame)

    # Update unique persons and weapons based on centroids
    update_unique_objects(unique_persons, persons, threshold=50)
    update_unique_objects(unique_weapons, weapons, threshold=50)

    # Identify person holding a weapon (if any)
    weapon_holder = identify_weapon_holder(weapons, persons)
    if weapon_holder is not None:
        px1, py1, px2, py2 = weapon_holder
        # Ensure coordinates are within the frame boundaries
        px1, py1 = max(px1, 0), max(py1, 0)
        px2, py2 = min(px2, frame_width), min(py2, frame_height)
        # Crop the region of the weapon holder from the frame
        crop_img = frame[py1:py2, px1:px2].copy()
        # Run pose estimation on the crop and save the annotated image
        weapon_holder_crop = run_pose_estimation_and_save(crop_img, frame_index)
        # The crop remains persistent until a new weapon holder is detected

    # Draw bounding boxes for persons (blue) and weapons (red)
    for (x1, y1, x2, y2) in persons:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
    for (x1, y1, x2, y2) in weapons:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
    if weapon_holder is not None:
        cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 255, 255), 2)  # Yellow for weapon holder

    # Display activity in the top-left corner in red
    cv2.putText(frame, f"Activity: {detected_activity}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)

    # Overlay persistent weapon holder crop (if available) in the top-right corner
    if weapon_holder_crop is not None:
        overlay = cv2.resize(weapon_holder_crop, (150, 150))
        frame[0:150, frame_width-150:frame_width] = overlay

    # Write the processed frame to the output video
    out.write(frame)

    # Optional: show the frame
    # cv2.imshow("Crime Detection", frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

cap.release()
out.release()
cv2.destroyAllWindows()

# Timestamp Integeration

In [14]:
import cv2
from datetime import datetime, timedelta

# ==================== Video Processing ====================
video_path = "/content/Burglary-1.mp4"  # Update with your video path
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
output_video_path = "/content/output_video.mp4"
fourcc = cv2.VideoWriter_fourcc(*'avc1')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# For unique detection we use lists to store the centroid of each unique detection.
unique_persons = []   # List of centers (x, y) for unique persons
unique_weapons = []   # List of centers (x, y) for unique weapons

# For activity counts (if needed)
activity_counts = {label: 0 for label in class_labels.values()}

# Variable for persistent weapon holder overlay:
weapon_holder_crop = None  # Stores the cropped & pose–annotated image of the person holding a weapon

# Store timestamps for detections
detection_timestamps = []
video_start_time = datetime.now()

frame_index = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_index += 1
    # Calculate real-time timestamp
    frame_time = video_start_time + timedelta(seconds=frame_index / fps)
    timestamp_str = frame_time.strftime("%Y-%m-%d %H:%M:%S")

    # Detect activity
    detected_activity = predict_activity(frame)
    activity_counts[detected_activity] += 1

    # Detect objects (weapons and persons)
    weapons, persons = detect_objects(frame)

    # Update unique persons and weapons based on centroids
    update_unique_objects(unique_persons, persons, threshold=50)
    update_unique_objects(unique_weapons, weapons, threshold=50)

    # Identify person holding a weapon (if any)
    weapon_holder = identify_weapon_holder(weapons, persons)
    if weapon_holder is not None:
        px1, py1, px2, py2 = weapon_holder
        px1, py1 = max(px1, 0), max(py1, 0)
        px2, py2 = min(px2, frame_width), min(py2, frame_height)
        crop_img = frame[py1:py2, px1:px2].copy()
        weapon_holder_crop = run_pose_estimation_and_save(crop_img, frame_index)

    # Store timestamps for detections (only if activity is not normal)
    if detected_activity != "normal" or weapon_holder is not None:
        detection_timestamps.append(timestamp_str)

    # Draw bounding boxes for persons (blue) and weapons (red)
    for (x1, y1, x2, y2) in persons:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
    for (x1, y1, x2, y2) in weapons:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
    if weapon_holder is not None:
        cv2.rectangle(frame, (px1, py1), (px2, py2), (0, 255, 255), 2)  # Yellow for weapon holder

    # Display activity in the top-left corner in red
    cv2.putText(frame, f"Activity: {detected_activity}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"Time: {timestamp_str}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    # Overlay persistent weapon holder crop (if available) in the top-right corner
    if weapon_holder_crop is not None:
        overlay = cv2.resize(weapon_holder_crop, (150, 150))
        frame[0:150, frame_width-150:frame_width] = overlay

    # Write the processed frame to the output video
    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()

# Save timestamps for frontend integration
with open("detection_timestamps.txt", "w") as f:
    for ts in detection_timestamps:
        f.write(ts + "\n")

print("Detection timestamps saved successfully!")



0: 384x640 3 Persons, 7 Pistols, 39.2ms
Speed: 10.0ms preprocess, 39.2ms inference, 332.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Persons, 7 Pistols, 10.7ms
Speed: 3.5ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Persons, 7 Pistols, 10.7ms
Speed: 3.2ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Persons, 7 Pistols, 9.8ms
Speed: 2.6ms preprocess, 9.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Persons, 7 Pistols, 9.7ms
Speed: 3.3ms preprocess, 9.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Persons, 7 Pistols, 20.1ms
Speed: 2.3ms preprocess, 20.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Persons, 7 Pistols, 10.6ms
Speed: 3.3ms preprocess, 10.6ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Persons, 7 Pistols, 9

In [ ]:
# At the end, the unique counts are based on the number of unique centroids recorded.
unique_person_count = len(unique_persons)
unique_weapon_count = len(unique_weapons)

# Determine the most frequently detected activity (if needed for the report)
most_frequent_activity = max(activity_counts, key=activity_counts.get)

# Generate detailed scene description using OpenAI (with FLAN-T5 fallback)
scene_description = generate_scene_description(most_frequent_activity, unique_person_count, unique_weapon_count)

# Generate the final report as a PDF with custom formatting
output_pdf_path = "/content/crime_scene_report.pdf"
generate_pdf_report(most_frequent_activity, scene_description, unique_person_count, unique_weapon_count, output_pdf_path)

print(f"Report generated: {output_pdf_path}")

Report generated: /content/crime_scene_report.pdf
